# Piloto físico na região CT Knobs

Este notebook conduz o piloto de incorporação de física à modelagem térmica, na região dos knobs do cooktop. A região foi escolhida por concentrar a maior fração de pontos governados por condução em sólido, quatro dos seis, e por ter geometria simples.

O piloto responde a uma pergunta: um modelo físico da condução ao longo do console melhora a previsão em relação ao modelo estatístico já entregue?

A resposta obtida é negativa, e a razão pela qual ela é negativa constitui o principal resultado deste notebook. A investigação da causa foi estendida às sete regiões, na Seção 6.

## Registro prévio

Nenhuma leitura desta região se aproxima do limite. O limite dos seis pontos é 105 graus e a maior leitura registrada é 73,4 graus, de modo que a região não participa de nenhuma decisão de aprovação. O valor do piloto é metodológico.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4/testes_extras/colabs_FEM"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

BASE = "../../"
PARTICAO = "../../../colabs_S3/preparacao_testes/particao_treino_teste.json"

PREFIXOS_ENTRADA = ("blower_direction_", "housing_", "board_position_", "run_case_",
                    "losses_", "perfil_perdas_", "deflector_hole_",
                    "material_coil_support_", "cfg_",
          "reason_", "loss_profile_order_")
PARAMS_RF = dict(n_estimators=300, random_state=42, n_jobs=-1)
PISO_RUIDO = 0.92   # graus, piso de ruido medido para a regiao

particao = json.load(open(PARTICAO, encoding="utf-8"))
DOBRA = {int(k): v for k, v in particao["dobra_por_run_idx"].items()}

dados = pd.read_csv(BASE + "dataset_ct_knobs_13cat.csv")
entradas = [c for c in dados.columns if c.startswith(PREFIXOS_ENTRADA)]
alvos = [c for c in dados.columns if not c.startswith(PREFIXOS_ENTRADA) and c != "run_idx"]

dados["_run"] = np.arange(len(dados))
dados["_dobra"] = dados["_run"].map(DOBRA)

print(f"ensaios na base: {len(dados)}")
print(f"colunas de entrada: {len(entradas)}")
print(f"componentes instrumentados: {len(alvos)}")
for c in alvos:
    print(f"   {c:<34} media {dados[c].mean():6.1f} C   desvio {dados[c].std():5.1f} C")

## 1. Tabela de suposições declaradas

As informações de geometria e de material ainda não foram recebidas da parceira. As suposições abaixo sustentam o modelo físico e estão registradas com origem e faixa testada, de modo que nenhum valor suposto seja apresentado como se fosse medido.

In [ ]:
suposicoes = pd.DataFrame([
    {"grandeza": "material do painel", "adotado": "aco inoxidavel",
     "origem": "vista explodida do material de kickoff, que mostra caixa metalica",
     "faixa testada": "polimero (k=0,20 a 0,29) contra aco (k=16)"},
    {"grandeza": "espessura do painel", "adotado": "3 mm",
     "origem": "valor tipico de chapa de eletrodomestico",
     "faixa testada": "2 a 4 mm"},
    {"grandeza": "coeficiente de conveccao", "adotado": "7,5 W/(m2.K)",
     "origem": "conveccao natural em painel vertical, Churchill-Chu",
     "faixa testada": "5 a 10"},
    {"grandeza": "espacamento entre knobs", "adotado": "100 a 130 mm",
     "origem": "dimensao tipica de fogao de 30 polegadas",
     "faixa testada": "nao aplicavel"},
    {"grandeza": "posicao dos termopares", "adotado": "desconhecida",
     "origem": "pendente junto a parceira",
     "faixa testada": "estimada a partir dos dados, Secao 3"},
])
print(suposicoes.to_string(index=False))

## 2. Linha de base

Duas referências são estabelecidas antes de qualquer proposta. A primeira é a média histórica de cada componente, calculada apenas sobre os ensaios de treino de cada dobra. A segunda é o modelo por área efetivamente entregue, um Random Forest em formato longo com patamar e escala do componente como entradas derivadas.

O protocolo é o mesmo da entrega: validação cruzada deixando uma configuração de fora por vez.

In [ ]:
completos = dados["_dobra"].notna() & dados[alvos].notna().all(axis=1)
d = dados[completos].reset_index(drop=True)
X = d[entradas].to_numpy(float)
Y = d[alvos].to_numpy(float)
dobras = d["_dobra"].to_numpy()
print(f"ensaios com dobra atribuida e leituras completas nos seis pontos: {len(d)}")
print(f"dobras distintas: {len(set(dobras))}")

In [ ]:
def ajusta_por_area(X_tr, Y_tr, X_va, media, desvio):
    """Random Forest unico da area, em formato longo, como na solucao entregue."""
    n_alvos = Y_tr.shape[1]
    emp_tr = np.vstack([np.hstack([X_tr,
                                   np.full((len(X_tr), 1), media[j]),
                                   np.full((len(X_tr), 1), desvio[j])])
                        for j in range(n_alvos)])
    y_tr = np.concatenate([Y_tr[:, j] for j in range(n_alvos)])
    emp_va = np.vstack([np.hstack([X_va,
                                   np.full((len(X_va), 1), media[j]),
                                   np.full((len(X_va), 1), desvio[j])])
                        for j in range(n_alvos)])
    escala = StandardScaler()
    modelo = RandomForestRegressor(**PARAMS_RF)
    modelo.fit(escala.fit_transform(emp_tr), y_tr)
    return modelo.predict(escala.transform(emp_va)).reshape(n_alvos, len(X_va)).T

## 3. A hipótese física e o que ela prevê

O console é tratado como uma aleta plana: uma tira aquecida por trás, que perde calor para o ar ao longo do comprimento. A solução conhecida desse problema é

T(x) = T_ar + (T_base - T_ar) * cosh[m(L - x)] / cosh(mL),   com m = raiz(2h / (k e))

O ponto relevante não é a fórmula, e sim a sua consequência estrutural. Se todos os pontos da região compartilham a mesma base e o mesmo ar, a temperatura de cada ponto é combinação de dois escalares por ensaio, a temperatura da base e a do ar, com coeficientes que dependem apenas da posição do ponto e são fixos entre ensaios.

Em outras palavras, a matriz de leituras, com ensaios nas linhas e componentes nas colunas, deve ter posto dois depois de removida a média de cada componente.

Essa é uma previsão verificável sem nenhum dado da parceira, e é o primeiro teste do piloto. Se a matriz não tiver posto baixo, a hipótese da aleta está errada e não vale prosseguir.

In [ ]:
centrada = Y - Y.mean(0)
U, S, Vt = np.linalg.svd(centrada, full_matrices=False)
variancia = np.cumsum(S ** 2) / (S ** 2).sum()

print("decomposicao da matriz de leituras, centrada por componente")
print(f"{'modos':>6}{'valor singular':>16}{'variancia acumulada':>22}{'MAE residual':>15}")
for k in range(len(S)):
    recon = U[:, :k] @ np.diag(S[:k]) @ Vt[:k] if k else np.zeros_like(centrada)
    print(f"{k:>6}{S[k]:>16.1f}{variancia[k-1] if k else 0:>22.4f}"
          f"{np.abs(centrada - recon).mean():>15.3f}")

O resultado sustenta a hipótese. Dois modos explicam a quase totalidade da variação entre componentes, e o erro residual da reconstrução com dois modos situa-se abaixo do piso de ruído da região.

Cabe interpretar os dois modos. O primeiro tem o mesmo sinal em todos os componentes e corresponde ao nível geral de temperatura da região. O segundo troca de sinal entre o lado esquerdo e o direito, e corresponde ao desequilíbrio entre os dois lados do console. São exatamente os dois graus de liberdade que a física prevê.

In [ ]:
perfil = pd.DataFrame({"componente": alvos,
                       "media (C)": Y.mean(0).round(1),
                       "modo 1 (nivel)": Vt[0].round(3),
                       "modo 2 (contraste)": Vt[1].round(3)})
print(perfil.to_string(index=False))
print()
print("O modo 2 separa os componentes em dois grupos de sinal oposto,")
print("o que corresponde ao desequilibrio esquerda-direita do console.")

## 4. Estudo de sensibilidade: qual incerteza de fato importa

Antes de calibrar, convém saber quais das grandezas desconhecidas alteram a resposta. O comprimento característico de decaimento, igual a 1/m, indica a distância em que o calor deixa de se propagar pelo painel. Comparado ao espaçamento entre knobs, ele decide se a condução ao longo do painel transporta calor entre pontos vizinhos.

In [ ]:
def comprimento_decaimento(k, espessura, h=7.5):
    """Devolve 1/m em milimetros, para aleta com conveccao nas duas faces."""
    return 1000.0 / np.sqrt(2 * h / (k * espessura))

materiais = {"PC/ABS": 0.20, "Nylon": 0.25, "PBT": 0.29,
             "Aco inoxidavel": 16.0, "Aluminio": 205.0}
print(f"{'material':<18}{'k':>8}{'2 mm':>9}{'3 mm':>9}{'4 mm':>9}")
for nome, k in materiais.items():
    v = [comprimento_decaimento(k, e) for e in (0.002, 0.003, 0.004)]
    print(f"{nome:<18}{k:>8.2f}" + "".join(f"{x:>9.1f}" for x in v))
print()
print("espacamento tipico entre knobs: 100 a 130 mm")
print()
print("sensibilidade da temperatura a 100 mm da base:")
L = 0.30
def T_aleta(k, h, espessura, x=0.10, T_ar=48.0, T_base=70.0):
    m = np.sqrt(2 * h / (k * espessura))
    return T_ar + (T_base - T_ar) * np.cosh(m * (L - x)) / np.cosh(m * L)
ref = T_aleta(0.25, 7.5, 0.003)
print(f"  referencia, polimero: {ref:.2f} C")
for rot, args in [("k de 0,20 a 0,29", [(0.20, 7.5, 0.003), (0.29, 7.5, 0.003)]),
                  ("h de 5 a 10", [(0.25, 5.0, 0.003), (0.25, 10.0, 0.003)]),
                  ("espessura de 2 a 4 mm", [(0.25, 7.5, 0.002), (0.25, 7.5, 0.004)]),
                  ("polimero contra aco", [(0.25, 7.5, 0.003), (16.0, 7.5, 0.003)])]:
    v = [T_aleta(*a) for a in args]
    print(f"  {rot:<24} amplitude {abs(v[1]-v[0]):.3f} C")

Três das quatro incertezas não movem a resposta. Apenas a natureza do material altera o resultado, e a vista explodida do material de kickoff favorece a hipótese metálica, o que mantém a região como candidata válida.

O estudo de sensibilidade cumpre aqui a sua função: reduz a cinco perguntas pendentes a uma só, e permite prosseguir sem aguardar as demais.

## 5. A formulação por modos, e a sua avaliação

A estrutura verificada na Seção 3 sugere uma formulação alternativa. Em lugar de prever seis temperaturas a partir da configuração, o modelo prevê dois escalares por ensaio, e as seis temperaturas são reconstruídas pelos perfis espaciais.

A redução é substancial: de seis alvos para dois, o que constitui exatamente o tipo de restrição que costuma ajudar em base pequena.

Três cuidados garantem que a medição seja honesta. Os perfis espaciais e as médias por componente são estimados apenas com os ensaios de treino de cada dobra. Os escalares de validação são previstos, e não observados. E um teto é calculado à parte, usando os escalares verdadeiros da validação, para separar o que a estrutura espacial resolve do que depende das entradas.

In [ ]:
registros = []
for k in sorted(set(dobras)):
    val = dobras == k
    tre = ~val
    if tre.sum() < 10 or not val.any():
        continue

    media = Y[tre].mean(0)
    desvio = Y[tre].std(0)
    desvio[desvio == 0] = 1.0

    pred_area = ajusta_por_area(X[tre], Y[tre], X[val], media, desvio)

    centro = Y[tre] - media
    _, _, V = np.linalg.svd(centro, full_matrices=False)
    perfis = V[:2]
    escalares_tre = centro @ perfis.T
    escalares_val = np.zeros((val.sum(), 2))
    for j in range(2):
        esc = StandardScaler()
        modelo_z = RandomForestRegressor(**PARAMS_RF)
        modelo_z.fit(esc.fit_transform(X[tre]), escalares_tre[:, j])
        escalares_val[:, j] = modelo_z.predict(esc.transform(X[val]))
    pred_modos = media + escalares_val @ perfis

    pred_teto = media + ((Y[val] - media) @ perfis.T) @ perfis

    for i, linha in enumerate(np.where(val)[0]):
        for j, comp in enumerate(alvos):
            registros.append({"dobra": k, "componente": comp, "real": Y[linha, j],
                              "baseline": media[j], "area": pred_area[i, j],
                              "modos": pred_modos[i, j], "teto": pred_teto[i, j]})

res = pd.DataFrame(registros)
for c in ("baseline", "area", "modos", "teto"):
    res["e_" + c] = (res[c] - res["real"]).abs()
print(f"previsoes de validacao: {len(res)}")

In [ ]:
print("=" * 72)
print("CT KNOBS, validacao cruzada por configuracao")
print("=" * 72)
print(f"{'formulacao':<36}{'MAE':>9}{'RMSE':>9}{'dentro 10%':>13}")
rotulos = [("baseline", "media do componente"),
           ("area", "modelo por area (entregue)"),
           ("modos", "dois modos espaciais (proposta)"),
           ("teto", "teto: escalares verdadeiros")]
for c, rot in rotulos:
    e = res["e_" + c]
    rel = e / res["real"]
    print(f"{rot:<36}{e.mean():>9.3f}{np.sqrt((e ** 2).mean()):>9.3f}"
          f"{100 * (rel <= 0.10).mean():>12.1f}%")
print()
print(f"piso de ruido da regiao: {PISO_RUIDO:.2f} C")
print()
dif = res["e_area"] - res["e_modos"]
print(f"teste pareado, area contra modos (positivo favorece a proposta)")
print(f"  diferenca media {dif.mean():+.4f} C")
print(f"  t de Student  p = {stats.ttest_rel(res['e_area'], res['e_modos']).pvalue:.3e}")
print(f"  Wilcoxon      p = {stats.wilcoxon(res['e_area'], res['e_modos']).pvalue:.3e}")
reamostras = [np.random.default_rng(s).choice(dif.to_numpy(), len(dif), replace=True).mean()
              for s in range(2000)]
print(f"  IC bootstrap 95%: [{np.percentile(reamostras, 2.5):+.3f}, "
      f"{np.percentile(reamostras, 97.5):+.3f}] C")

## 6. O resultado, e o que ele isola

A formulação por modos não supera o modelo por área. A diferença é pequena e o intervalo de confiança contém o zero, de modo que não há efeito a reportar.

O resultado decisivo, contudo, é o teto. Conhecidos os dois escalares de cada ensaio, o erro cai para menos de um grau, abaixo do piso de ruído da região. Isso significa que a estrutura espacial entre os componentes está essencialmente resolvida, e que a dificuldade não está na forma do modelo.

A dificuldade está em prever os dois escalares a partir da configuração. Ou seja, o que falta é informação nas entradas, e não sofisticação no modelo. Nenhum modelo físico da condução ao longo do console mudaria isso, porque o console não é o problema.

Convém verificar se a conclusão vale apenas para esta região ou para o projeto inteiro. O teste é barato e foi estendido às sete.

In [ ]:
PISOS = {"oven_hmi": 0.58, "ct_knobs": 0.92, "main_vent_sys": 1.16, "ipc_cooling": 2.15,
         "mainback": 1.33, "ipc_comp": 3.30, "coil_temps": 6.36}
ATUAL = {"coil_temps": 11.71, "ct_knobs": 3.16, "ipc_comp": 7.30, "ipc_cooling": 4.04,
         "main_vent_sys": 4.68, "mainback": 5.68, "oven_hmi": 2.42}

linhas = []
for area in PISOS:
    da = pd.read_csv(BASE + f"dataset_{area}_13cat.csv")
    col = [c for c in da.columns if not c.startswith(PREFIXOS_ENTRADA) and c != "run_idx"]
    M = da[col].dropna()
    if len(M) < 10:
        continue
    A = M.to_numpy(float)
    Cc = A - A.mean(0)
    Uu, Ss, Vv = np.linalg.svd(Cc, full_matrices=False)
    k95 = int(np.searchsorted(np.cumsum(Ss ** 2) / (Ss ** 2).sum(), 0.95) + 1)
    rec2 = Uu[:, :2] @ np.diag(Ss[:2]) @ Vv[:2]
    teto2 = np.abs(Cc - rec2).mean()
    linhas.append({"area": area, "ensaios completos": len(M), "componentes": len(col),
                   "modos p/ 95%": k95, "teto k=2": teto2, "piso": PISOS[area],
                   "erro atual": ATUAL[area],
                   "% do erro fora da estrutura espacial": 100 * (ATUAL[area] - teto2) / ATUAL[area]})

diag = pd.DataFrame(linhas).sort_values("% do erro fora da estrutura espacial", ascending=False)
print(diag.round(2).to_string(index=False))

## 7. Comparação final e veredito

As seções anteriores mediram a proposta física desta região. Esta seção a confronta com as demais formulações disponíveis, todas avaliadas sobre as mesmas leituras e as mesmas dobras, e registra o desfecho de cada caminho.

Quatro formulações são comparadas.

O preditor de referência prevê para cada componente a média histórica calculada sobre os ensaios de treino da própria dobra.

A formulação entregue é o Random Forest por área, com a codificação de perfil da coluna de perdas.

A formulação por modos é a proposta física deste piloto, que prevê dois escalares por ensaio e reconstrói as seis leituras pelos perfis espaciais estimados.

A formulação final é a adotada ao término da investigação: codificação decomposta da coluna de perdas, com média entre Random Forest e Gradient Boosting. Ela não decorre de física, e sim da reorganização de informação já presente na base.

O teto permanece como referência, e não como formulação: ele usa os escalares observados da validação, de modo que não constitui previsão.

Registra-se, por fim, que nenhuma das formulações cria dado novo. Este piloto não gerou nenhuma linha sintética, e trabalha sobre os mesmos 55 ensaios da base.

In [ ]:
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor

d13 = pd.read_csv(BASE + "dataset_ct_knobs_13cat.csv")
cols13 = [c for c in d13.columns if c.startswith("losses_")]
rot = []
for i in range(len(d13)):
    marc = [c[len("losses_"):] for c in cols13 if d13.at[i, c] == 1]
    rot.append(marc[0] if marc else "")
rot = pd.Series(rot)


def familia(r):
    if r == "":
        return "sem rotulo"
    if "no losses" in r:
        return "no losses"
    if "simone" in r:
        return "simone"
    if "miro" in r:
        return "miro"
    return "outro"


def ordem_acionamento(r):
    if r == "" or "no losses" in r:
        return "nao se aplica"
    if "front order" in r:
        return "front"
    if "back order" in r:
        return "back"
    return "nao declarada"


dec = np.hstack([
    pd.get_dummies(rot.map(familia)).to_numpy(float),
    pd.get_dummies(rot.map(ordem_acionamento)).to_numpy(float),
    rot.str.contains("modified").astype(float).to_numpy().reshape(-1, 1),
])[completos.to_numpy()]
comuns = [c for c in dados.columns
          if c.startswith(PREFIXOS_ENTRADA) and not c.startswith(("losses_", "perfil_perdas_"))]
X_dec = np.hstack([d[comuns].to_numpy(float), dec])

registros_final = []
for k in sorted(set(dobras)):
    val = dobras == k
    tre = ~val
    if tre.sum() < 10 or not val.any():
        continue
    media = Y[tre].mean(0)
    desvio = Y[tre].std(0)
    desvio[desvio == 0] = 1.0

    pred_area = ajusta_por_area(X[tre], Y[tre], X[val], media, desvio)

    centro = Y[tre] - media
    _, _, V = np.linalg.svd(centro, full_matrices=False)
    perfis = V[:2]
    z_tre = centro @ perfis.T
    z_val = np.zeros((val.sum(), 2))
    for j in range(2):
        e = StandardScaler()
        m = RandomForestRegressor(**PARAMS_RF).fit(e.fit_transform(X[tre]), z_tre[:, j])
        z_val[:, j] = m.predict(e.transform(X[val]))
    pred_modos = media + z_val @ perfis

    n_alvos = Y.shape[1]
    emp_tre = np.vstack([np.hstack([X_dec[tre],
                                    np.full((tre.sum(), 1), media[j]),
                                    np.full((tre.sum(), 1), desvio[j])])
                         for j in range(n_alvos)])
    y_tre = np.concatenate([Y[tre, j] for j in range(n_alvos)])
    emp_val = np.vstack([np.hstack([X_dec[val],
                                    np.full((val.sum(), 1), media[j]),
                                    np.full((val.sum(), 1), desvio[j])])
                         for j in range(n_alvos)])
    e = StandardScaler()
    Xt, Xv = e.fit_transform(emp_tre), e.transform(emp_val)
    p_rf = RandomForestRegressor(**PARAMS_RF).fit(Xt, y_tre).predict(Xv)
    p_gb = GradientBoostingRegressor(n_estimators=300, random_state=42).fit(Xt, y_tre).predict(Xv)
    pred_final = ((p_rf + p_gb) / 2).reshape(n_alvos, val.sum()).T

    pred_teto = media + ((Y[val] - media) @ perfis.T) @ perfis

    for i, linha in enumerate(np.where(val)[0]):
        for j, comp in enumerate(alvos):
            registros_final.append({"componente": comp, "real": Y[linha, j],
                                    "referencia": media[j], "entregue": pred_area[i, j],
                                    "modos": pred_modos[i, j], "final": pred_final[i, j],
                                    "teto": pred_teto[i, j]})

comp_final = pd.DataFrame(registros_final)
for col in ("referencia", "entregue", "modos", "final", "teto"):
    comp_final["e_" + col] = (comp_final[col] - comp_final["real"]).abs()
print(f"previsoes comparadas: {len(comp_final)}")

In [ ]:
rotulos = [("referencia", "media historica do componente"),
           ("entregue", "modelo por area, como entregue"),
           ("modos", "dois modos espaciais, proposta fisica deste piloto"),
           ("final", "codificacao decomposta com media RF e GB"),
           ("teto", "teto: escalares verdadeiros, nao e previsao")]
print("=" * 84)
print("CT KNOBS, todas as formulacoes sobre as mesmas leituras")
print("=" * 84)
print(f"{'formulacao':<46}{'MAE':>8}{'RMSE':>8}{'dentro 10%':>12}")
for col, rot_ in rotulos:
    e = comp_final["e_" + col]
    rel = e / comp_final["real"]
    print(f"{rot_:<46}{e.mean():>8.3f}{np.sqrt((e ** 2).mean()):>8.3f}"
          f"{100 * (rel <= 0.10).mean():>11.1f}%")
print()
print(f"piso de ruido da regiao: {PISO_RUIDO:.2f} C")
print()
print("VEREDITO DE CADA CAMINHO, contra a formulacao entregue")
print("-" * 84)
for col, rot_ in [("modos", "proposta fisica (dois modos)"),
                  ("final", "reorganizacao da coluna de perdas")]:
    dif = comp_final["e_entregue"] - comp_final["e_" + col]
    amostras = [np.random.default_rng(s).choice(dif.to_numpy(), len(dif), replace=True).mean()
                for s in range(3000)]
    lo, hi = np.percentile(amostras, [2.5, 97.5])
    p = stats.wilcoxon(comp_final["e_entregue"], comp_final["e_" + col]).pvalue
    if lo > 0:
        v = "MELHOROU"
    elif hi < 0:
        v = "PIOROU"
    else:
        v = "INEFICAZ (efeito nao distinguivel de zero)"
    print(f"  {rot_:<38} ganho {dif.mean():+.3f} C | IC95 [{lo:+.3f}, {hi:+.3f}] | "
          f"p={p:.4f}")
    print(f"  {'':<38} -> {v}")
print()
print("dados novos gerados por este piloto: 0")
print(f"ensaios utilizados: {len(d)} | leituras: {len(comp_final)}")

O confronto separa dois caminhos que este piloto percorreu.

O caminho físico, que reformula o problema a partir da estrutura de aleta, não se distingue da formulação entregue. A proposta é ineficaz, e a razão está registrada na seção anterior: a estrutura espacial já se encontra resolvida, e o que limita a previsão é a informação contida nas variáveis de configuração.

O caminho da reorganização da informação, por sua vez, melhora. Ele não decorre de física nem de dado novo, e sim da observação de que os rótulos da coluna de perdas são compostos e de que a ordem de acionamento das bobinas encontrava-se dissolvida dentro deles.

A leitura conjunta é a lição deste piloto. Na região dos knobs, e conforme o diagnóstico da seção anterior em seis das sete regiões, o ganho não veio de modelar melhor o fenômeno, e sim de extrair melhor o que a base já registrava.

## 8. O método dos elementos finitos aplicado a esta região

As seções anteriores empregaram a solução analítica da aleta, que é exata para uma tira de espessura constante. Esta seção substitui essa solução por um resolvedor de elementos finitos em duas dimensões, e mede o que se ganha com a troca.

A pergunta não é se o método funciona, e sim se a geometria suposta descreve a peça. Um resolvedor correto sobre geometria errada produz números precisos e inúteis, e é essa distinção que a seção pretende separar.

O console é tratado como chapa plana em regime permanente, com condução no plano, perda convectiva pelas duas faces e pelas arestas laterais, e temperatura imposta na borda inferior, que representa o contato com a caixa metálica que abriga a placa de indução.

A propriedade que torna o resolvedor útil é a linearidade. Para um dado coeficiente de convecção, a solução é afim na temperatura da base e na do ar, de modo que um único ajuste fornece o perfil espacial e qualquer par de temperaturas é obtido sem resolver novamente. Esse perfil é exatamente o objeto que a decomposição da Seção 3 estimava a partir dos dados; aqui ele provém da geometria.

In [ ]:
from collections import Counter

LARGURA = 0.76      # m, console de fogao de 30 polegadas
ALTURA = 0.10       # m, faixa frontal
ESPESSURA = 0.003   # m, chapa
K_ACO = 16.0        # W/(m.K), inoxidavel

SONDAS = {
    "Left Encoder": (0.10, 0.055),
    "Ambient Between Left Encoders": (0.16, 0.055),
    "Left Center Encoder": (0.22, 0.055),
    "Right Center Encoder": (0.54, 0.055),
    "Ambient Between Right Encoders": (0.60, 0.055),
    "Right Encoder": (0.66, 0.055),
}


def malha(nx, ny, largura=LARGURA, altura=ALTURA):
    """Malha estruturada de triangulos sobre o retangulo do console."""
    xs = np.linspace(0, largura, nx + 1)
    ys = np.linspace(0, altura, ny + 1)
    X, Y_ = np.meshgrid(xs, ys, indexing="ij")
    p = np.column_stack([X.ravel(), Y_.ravel()])
    pos = lambda i, j: i * (ny + 1) + j
    t = []
    for i in range(nx):
        for j in range(ny):
            a, b, c, e = pos(i, j), pos(i + 1, j), pos(i + 1, j + 1), pos(i, j + 1)
            t.append([a, b, c])
            t.append([a, c, e])
    return p, np.array(t)


def monta(p, t, h):
    """Matriz de conducao com perda pelas faces, e contribuicoes de contorno."""
    n = len(p)
    A = np.zeros((n, n))
    fator = np.zeros(n)
    for tri in t:
        x, y_ = p[tri, 0], p[tri, 1]
        area = 0.5 * ((x[1] - x[0]) * (y_[2] - y_[0]) - (x[2] - x[0]) * (y_[1] - y_[0]))
        gx = np.array([y_[1] - y_[2], y_[2] - y_[0], y_[0] - y_[1]]) / (2 * area)
        gy = np.array([x[2] - x[1], x[0] - x[2], x[1] - x[0]]) / (2 * area)
        A[np.ix_(tri, tri)] += K_ACO * ESPESSURA * area * (np.outer(gx, gx) + np.outer(gy, gy))
        massa = area / 12.0 * (np.ones((3, 3)) + np.eye(3))
        A[np.ix_(tri, tri)] += 2 * h * massa          # perda pelas duas faces
        fator[tri] += 2 * h * area / 3.0              # carga do ar pelas faces
    arestas = []
    for tri in t:
        for a, c in ((tri[0], tri[1]), (tri[1], tri[2]), (tri[2], tri[0])):
            arestas.append((min(a, c), max(a, c)))
    contorno = [e for e, q in Counter(arestas).items() if q == 1]
    base = set()
    for a, c in contorno:
        if abs(p[a, 1]) < 1e-12 and abs(p[c, 1]) < 1e-12:
            base.update([a, c])
            continue
        L = np.hypot(p[c, 0] - p[a, 0], p[c, 1] - p[a, 1])
        A[np.ix_([a, c], [a, c])] += h * ESPESSURA * L * np.array([[1/3, 1/6], [1/6, 1/3]])
        fator[[a, c]] += h * ESPESSURA * L / 2        # carga do ar pelas arestas
    return A, fator, np.array(sorted(base))


def resolve(nx, ny, h, T_base, T_ar, largura=LARGURA, altura=ALTURA):
    """Campo de temperatura com a borda inferior a T_base e o ar a T_ar."""
    p, t = malha(nx, ny, largura, altura)
    A, fator, base = monta(p, t, h)
    b = fator * T_ar
    livres = np.setdiff1d(np.arange(len(p)), base)
    u = np.zeros(len(p))
    u[base] = T_base
    u[livres] = np.linalg.solve(A[np.ix_(livres, livres)],
                                b[livres] - A[np.ix_(livres, base)] @ u[base])
    return p, t, u


def interpola(p, t, u, ponto):
    """Valor do campo num ponto, por coordenadas baricentricas."""
    px, py = ponto
    for tri in t:
        x, y_ = p[tri, 0], p[tri, 1]
        det = (y_[1] - y_[2]) * (x[0] - x[2]) + (x[2] - x[1]) * (y_[0] - y_[2])
        l0 = ((y_[1] - y_[2]) * (px - x[2]) + (x[2] - x[1]) * (py - y_[2])) / det
        l1 = ((y_[2] - y_[0]) * (px - x[2]) + (x[0] - x[2]) * (py - y_[2])) / det
        if min(l0, l1, 1 - l0 - l1) >= -1e-9:
            return l0 * u[tri[0]] + l1 * u[tri[1]] + (1 - l0 - l1) * u[tri[2]]
    return float("nan")


print("resolvedor definido")

### 8.1. Verificação do resolvedor

Duas verificações precedem qualquer uso. A primeira refina a malha e observa se a resposta estabiliza; se continuar se movendo, o resultado é do tamanho do triângulo e não da física. A segunda confronta o resolvedor com a solução analítica da aleta numa faixa estreita, em que as duas devem concordar.

In [ ]:
print("VERIFICACAO 1: convergencia de malha")
print(f"{'nx x ny':>10}{'nos':>8}{'elementos':>11}{'T em x=0,22 m':>16}{'variacao':>11}")
anterior = None
for nx, ny in [(19, 3), (38, 5), (76, 10), (152, 20)]:
    p, t, u = resolve(nx, ny, 7.5, 70.0, 48.0)
    T = interpola(p, t, u, SONDAS["Left Center Encoder"])
    dif = "" if anterior is None else f"{abs(T - anterior):>11.4f}"
    print(f"{nx:>6} x {ny:<3}{len(p):>8}{len(t):>11}{T:>16.4f}{dif}")
    anterior = T

print()
print("VERIFICACAO 2: controle analitico numa faixa estreita")
h = 7.5
m = np.sqrt(2 * h / (K_ACO * ESPESSURA))
print(f"  comprimento caracteristico 1/m = {1000 / m:.1f} mm")
p, t, u = resolve(4, 60, h, 70.0, 48.0, largura=0.02, altura=0.30)
for y_ in (0.05, 0.10, 0.20):
    fem = interpola(p, t, u, (0.01, y_))
    ana = 48.0 + (70.0 - 48.0) * np.cosh(m * (0.30 - y_)) / np.cosh(m * 0.30)
    print(f"  y={y_ * 1000:>5.0f} mm   FEM {fem:>7.3f}   analitico {ana:>7.3f}   "
          f"diferenca {abs(fem - ana):.4f} C")
print()
print("A diferenca remanescente decorre de a faixa possuir largura finita, com")
print("perda tambem pelas arestas laterais, o que a aleta unidimensional ignora.")

### 8.2. Primeira tentativa, com fonte uniforme

Com a borda inferior a uma única temperatura, o perfil espacial obtido nas seis sondas revela imediatamente um problema.

In [ ]:
def perfil(h, nx=76, ny=10):
    """Fracao do excesso de temperatura que sobra em cada sonda."""
    p, t, u = resolve(nx, ny, h, T_base=1.0, T_ar=0.0)
    return np.array([interpola(p, t, u, SONDAS[s]) for s in SONDAS])

phi_natural, phi_soprador = perfil(8.0), perfil(45.0)
print(f"{'sonda':<34}{'h=8':>12}{'h=45':>12}")
for i, s in enumerate(SONDAS):
    print(f"{s:<34}{phi_natural[i]:>12.5f}{phi_soprador[i]:>12.5f}")
print()
print(f"amplitude do perfil com h=8: {phi_natural.max() - phi_natural.min():.5f}")
print("As seis sondas recebem praticamente o mesmo valor, porque estao todas a")
print("mesma altura da borda quente e a fonte foi suposta uniforme ao longo dela.")
print("O modelo preve seis temperaturas iguais; as leituras variam 17,7 graus.")

### 8.3. Segunda tentativa, com fonte não uniforme

A fonte passa a variar linearmente da esquerda para a direita, o que exige dois escalares em lugar de um. A solução torna-se afim em três grandezas por ensaio: a temperatura do ar e as temperaturas da base à esquerda e à direita.

A comparação que interessa é de representabilidade, e não de previsão. Concedem-se ao modelo físico os escalares verdadeiros de cada ensaio, e mede-se o quanto ele reproduz das leituras observadas. Um modelo que não descreve o passado não prevê o futuro.

Para que a comparação seja justa, as duas bases recebem os mesmos graus de liberdade: a média de cada componente, mais dois escalares por ensaio.

In [ ]:
def base_fem(h, nx=76, ny=10):
    """Duas formas: base quente so a esquerda e so a direita."""
    p, t = malha(nx, ny)
    A, _, base = monta(p, t, h)
    livres = np.setdiff1d(np.arange(len(p)), base)
    formas = []
    fracao = p[base, 0] / LARGURA
    for lado in ("esquerda", "direita"):
        u = np.zeros(len(p))
        u[base] = (1 - fracao) if lado == "esquerda" else fracao
        u[livres] = np.linalg.solve(A[np.ix_(livres, livres)],
                                    -A[np.ix_(livres, base)] @ u[base])
        formas.append(np.array([interpola(p, t, u, SONDAS[s]) for s in SONDAS]))
    return np.vstack(formas)


formas = base_fem(8.0)
print("formas do FEM nas seis sondas, com h=8:")
print(f"  esquerda {np.round(formas[0], 4)}")
print(f"  direita  {np.round(formas[1], 4)}")
print()

ordem_sondas = list(SONDAS)
Y_s = d[ordem_sondas].to_numpy(float)
media_c = Y_s.mean(0)
centrada_s = Y_s - media_c

G = np.vstack([formas[0] - formas[0].mean(), formas[1] - formas[1].mean()])
proj_fem = centrada_s @ np.linalg.pinv(G) @ G
Us, Ss, Vs = np.linalg.svd(centrada_s, full_matrices=False)
proj_dados = Us[:, :2] @ np.diag(Ss[:2]) @ Vs[:2]

print("REPRESENTABILIDADE, ambas as bases com os mesmos graus de liberdade")
print(f"{'base':<34}{'MAE residual':>14}{'variancia explicada':>22}")
for rot, pr in [("base dos dados (2 modos)", proj_dados),
                ("base do FEM (2 formas)", proj_fem),
                ("nada alem da media por componente", np.zeros_like(centrada_s))]:
    var = 100 * (np.linalg.norm(pr) ** 2) / (np.linalg.norm(centrada_s) ** 2)
    print(f"{rot:<34}{np.abs(centrada_s - pr).mean():>14.3f}{var:>21.1f}%")
print()
print(f"piso de ruido da regiao: {PISO_RUIDO:.2f} C")
print()
print("alinhamento entre as formas do FEM e os modos dos dados:")
for i, lado in enumerate(("esquerda", "direita")):
    v = formas[i] - formas[i].mean()
    v = v / np.linalg.norm(v)
    for j in range(2):
        w = Vs[j] / np.linalg.norm(Vs[j])
        print(f"  forma {lado:<9} x modo {j + 1}: {abs(float(v @ w)):.3f}")

### 8.4. Veredito sobre o método dos elementos finitos nesta região

O resolvedor está correto. Ele converge com o refino da malha e concorda com a solução analítica no caso de controle, com diferença atribuível à largura finita da faixa.

A geometria suposta, porém, não descreve a peça. Concedidos os escalares verdadeiros de cada ensaio, a base derivada do FEM explica cerca de um quinto da variação entre componentes, contra quase a totalidade explicada pela base estimada dos próprios dados, e o resíduo resultante aproxima-se do que se obteria sem base alguma.

O alinhamento entre as formas indica onde reside o erro. O modelo físico reproduz razoavelmente o segundo modo, que corresponde ao desequilíbrio entre os lados do console, o que era esperado, uma vez que a fonte foi parametrizada como variando lateralmente. Ele erra, contudo, o primeiro modo, que é o dominante e descreve as magnitudes relativas entre os seis pontos.

A conclusão é que os seis pontos não podem estar todos à mesma distância da fonte, como a geometria suposta admite. A dispersão observada exige distâncias distintas, caminhos térmicos próprios por trás de cada encoder, ou fonte concentrada em lugar de distribuída. Distinguir entre as três hipóteses depende da informação pendente junto à parceira, que é a posição dos termopares.

Substituir a formulação atual pelo método dos elementos finitos, com a geometria hoje disponível, degradaria a capacidade de representação da região. A substituição não é recomendada enquanto a geometria não for confirmada, e o resolvedor permanece registrado neste notebook para ser reaproveitado quando ela chegar.

## 9. Conclusões

O piloto produziu quatro resultados.

A hipótese física da aleta é sustentada pelos dados. A matriz de leituras da região tem posto dois depois de removida a média de cada componente, e os dois modos correspondem ao nível geral e ao desequilíbrio entre os lados do console, que são os graus de liberdade que a física prevê.

O estudo de sensibilidade reduziu cinco perguntas pendentes junto à parceira a uma só, a natureza do material do painel, e a vista explodida do material de kickoff favorece a hipótese metálica.

A reformulação por modos não melhora a previsão. A diferença em relação ao modelo entregue não é distinguível de zero.

O gargalo foi localizado. Com os escalares conhecidos, o erro cai abaixo do piso de ruído da região, o que indica que a estrutura espacial não é a dificuldade. Estendido às sete regiões, o diagnóstico mostra que em seis delas a maior parte do erro atual não decorre da estrutura espacial entre componentes, e sim da incapacidade de prever o nível a partir das variáveis de configuração disponíveis. A exceção é a região das bobinas, única em que a estrutura espacial exige três modos e em que o teto permanece próximo do erro atual.

A consequência prática é que o esforço de modelagem física tem retorno esperado na região das bobinas, e que nas demais o retorno depende de obter a grandeza ausente, que é a potência associada a cada perfil de perda.

## 10. Limitações

O teto é calculado com os escalares observados na própria validação, e constitui limite inferior do erro alcançável, não uma previsão. Ele serve para separar as duas parcelas da dificuldade, e não para estimar desempenho.

A decomposição exige leituras completas em todos os componentes da região, o que reduz o número de ensaios utilizáveis. Na região IPC Comp restam 22 ensaios completos dos 56, de modo que o diagnóstico dessa região apoia-se em base estreita e deve ser confirmado.

As suposições de geometria e material permanecem declaradas e não confirmadas, conforme a tabela da Seção 1. A confirmação do material do painel é a única pendência que altera o modelo físico desta região.